# Week 10 — Home exercise 2: Which variable, and in what shape

**Solution proposal.**

Twelve models in one loop, and the discovery that the lecture spent an hour on the second-best
variable.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

plt.style.use("ggplot")

cars = pd.read_excel("../data/mpg.xlsx").dropna(subset=["horsepower"])

print("rows:", len(cars))

rows: 392


## 2. A function that returns one number

In [2]:
def get_adj_r2(data, formula):
    """
    Fit an OLS model and return its adjusted R-squared.

    Parameters
    ----------
    data : DataFrame
        The data to fit the model on.
    formula : str
        An R-style model formula, for example "y ~ x1 + x2".

    Returns
    -------
    float
        The adjusted R-squared of the fitted model.
    """
    return smf.ols(formula, data=data).fit().rsquared_adj

In [3]:
print(round(get_adj_r2(cars, "mpg ~ horsepower"), 4))

0.6049


## 3. Four attributes, three shapes each

The three shapes differ only in how the predictor appears in the formula string, so the formulas can
be built from the attribute name with f-strings.

In [4]:
attributes = ["horsepower", "weight", "acceleration", "model_year"]

rows = []
for attribute in attributes:
    rows.append({
        "attribute": attribute,
        "straight line": get_adj_r2(cars, f"mpg ~ {attribute}"),
        "polynomial": get_adj_r2(cars, f"mpg ~ {attribute} + I({attribute}**2)"),
        "log": get_adj_r2(cars, f"mpg ~ np.log({attribute})"),
    })

shapes = pd.DataFrame(rows).set_index("attribute").round(3)

shapes

,straight line,polynomial,log
attribute,,,
horsepower,0.605,0.686,0.667
weight,0.692,0.714,0.712
acceleration,0.177,0.190,0.188
model_year,0.335,0.364,0.331


## 4. What the table says

**`weight` wins, and the lecture used `horsepower`.**

A straight line in `weight` alone gets adjusted R² of **0.692** — better than the *best* horsepower
model in the lecture, which was the polynomial at 0.686. Bending it helps a little more, to **0.714**.

Two other things worth reading off:

- **The shape matters most where the relationship is most curved.** For `horsepower`, going from a
  straight line to a polynomial buys 0.081. For `weight` it buys 0.022, and for `acceleration`
  practically nothing. A curve is not an improvement you apply everywhere; it is an improvement you
  apply where the residuals asked for one.
- **`acceleration` explains almost nothing** — 0.19 at best. It is in the file, it is numeric, and it
  is nearly useless here. Having a column is not a reason to model it.

And a caution about how close two of these are:

In [5]:
print("weight, polynomial:", round(shapes.loc["weight", "polynomial"], 4))
print("weight, log:       ", round(shapes.loc["weight", "log"], 4))

weight, polynomial: 0.714
weight, log:        0.712


0.714 against 0.712. **That is not a difference.** Picking the polynomial because it won by 0.002
would be reading noise as evidence — and, as the lecture showed, the polynomial is the one that turns
around and misbehaves outside the data. On a gap that size, the choice should be made on which shape
you can defend, not on the third decimal.

## 5. `cylinders`: a number or a label?

In [6]:
cars["cylinders"].value_counts().sort_index()

cylinders
3      4
4    199
5      3
6     83
8    103
Name: count, dtype: int64

In [7]:
as_number = get_adj_r2(cars, "mpg ~ horsepower + cylinders")
as_category = get_adj_r2(cars, "mpg ~ horsepower + C(cylinders)")

print("cylinders as a number  :", round(as_number, 4))
print("cylinders as a category:", round(as_category, 4))

cylinders as a number  : 0.6551
cylinders as a category: 0.7008


**0.655 against 0.701**, so treating `cylinders` as a label is worth about 4.6 points of adjusted R².

As a *number*, the model is forced to assume that every extra cylinder does the same thing: the step
from 4 to 5 must equal the step from 7 to 8. As a *category*, each value gets its own intercept and no
such assumption is made — which is why it fits better.

### Would I use it?

**Here, yes**, with one reservation. Four values out of five are well populated: 199 four-cylinder
cars, 103 eight-cylinder, 83 six-cylinder. Those three carry the result.

The reservation is the other two. **Four cars have 3 cylinders and three cars have 5.** Each of those
gets a coefficient of its own, estimated from a handful of rows, and it will be a large number with a
large standard error attached. The model is not wrong to do it; you just should not quote those two
coefficients as if they meant anything.

The cost of the categorical version is interpretability. The numeric version gives you one number to
put in a sentence — *"each extra cylinder costs so many mpg"*. The categorical version gives you four
numbers that only mean anything against a baseline you also have to explain. With five categories that
is a fair trade. With fifty it would not be, and with a continuous variable it would be absurd.

## 6. The best two together

In [8]:
print("weight alone         :", round(get_adj_r2(cars, "mpg ~ weight"), 4))
print("horsepower alone     :", round(get_adj_r2(cars, "mpg ~ horsepower"), 4))
print("both                 :", round(get_adj_r2(cars, "mpg ~ weight + horsepower"), 4))
print()
print("correlation between them:", round(cars["weight"].corr(cars["horsepower"]), 3))

weight alone         : 0.6918
horsepower alone     : 0.6049
both                 : 0.7049

correlation between them: 0.865


**0.692 and 0.605 separately, 0.705 together** — the second variable adds 0.013.

That is a very small gain for a whole extra variable, and the correlation of **0.865** explains it.
Weight and horsepower are close to the same measurement taken two ways: heavy cars have big engines.
Adding the second one tells the model almost nothing it did not already know.

This is the same fact the lecture met from the other side, when the horsepower coefficient collapsed
from -0.158 to -0.047 once weight was included. Two variables that overlap that much will always
behave like this: the fit barely improves, and the coefficients become hard to interpret separately,
because there are very few cars in this data that are heavy with a small engine or light with a big
one.

### Things worth noticing

- **A one-line function turned twelve models into a table.** Not because fitting is hard, but because
  `get_adj_r2(cars, f"mpg ~ {attribute}")` inside a loop is readable and twelve pasted cells are not.
- **f-strings build formulas.** `f"mpg ~ {attribute} + I({attribute}**2)"` is the whole trick behind
  the specification loops in the lecture, applied to the variable rather than the model.
- **Every model in the table explains `mpg` on the same 392 rows**, which is what makes the twelve
  numbers comparable. Had one of them logged the *outcome*, it would have belonged in a different
  table.

### What this notebook does NOT do

- **It never looks at a residual plot.** Adjusted R² is one number per model, and the whole lesson of
  the lecture's section 3 is that one number cannot tell you whether the shape is right. A model that
  wins this table can still be wrong in a visible, systematic way.
- **It compares shapes one variable at a time.** The best model here is almost certainly not the best
  single attribute in the best shape — it is some combination — and nothing here searches for that.
- **Adjusted R² is being used to choose a model**, which is exactly what the lecture warned against
  doing on its own. The choices survive here because the gaps are large; on the 0.714-against-0.712
  comparison in question 4, the number decided nothing and judgment had to.